# 01 — Data Quality Assessment

## Superstore Sales Analytics

Este notebook tem como objetivo realizar uma auditoria inicial da qualidade dos dados antes das etapas de transformação e análise.

### Objetivos
- Compreender a estrutura e a granularidade do dataset
- Validar tipos de dados
- Identificar valores ausentes
- Identificar duplicidades
- Avaliar cardinalidade
- Validar possíveis chaves
- Verificar consistência de datas
- Identificar valores e regras de negócio que necessitem tratamento

> O dataset localizado em data/raw representa a fonte original e não será alterado.

## 1. Configuração e Fonte de Dados

### 1.1 — Configuração do Ambiente e Caminhos

A auditoria inicia pela configuração dos recursos necessários para leitura e manipulação dos dados.

Path é utilizado para construir os caminhos do projeto de forma independente de diretórios absolutos da máquina, enquanto pandas fornece as estruturas utilizadas nas validações de qualidade subsequentes.

A raiz do projeto é determinada programaticamente a partir do diretório do notebook. A partir dela, são definidos os diretórios das camadas raw e processed, mantendo separadas a fonte original e os dados que serão posteriormente tratados.

In [32]:
from pathlib import Path

import pandas as pd

In [33]:
PROJECT_ROOT = Path.cwd().parent

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

DATA_RAW

WindowsPath('c:/superstore-analytics/data/raw')

### 1.2 — Carregamento da Fonte Bruta

O arquivo sales_data.xlsx é carregado diretamente da camada data/raw, sem aplicação de transformações prévias.

O DataFrame representa o estado original da fonte durante toda a auditoria. A quantidade inicial de linhas e colunas é registrada como referência para as validações posteriores e para comparação com a camada processada.

In [34]:
file_path = DATA_RAW / "sales_data.xlsx"

df = pd.read_excel(file_path)

print("Dataset carregado com sucesso.")
print(f"Linhas: {df.shape[0]:,}")
print(f"Colunas: {df.shape[1]}")

Dataset carregado com sucesso.
Linhas: 9,800
Colunas: 19


## 2. Diagnóstico Estrutural do Dataset

### 2.1 — Amostra Inicial

A inspeção inicial permite verificar visualmente a estrutura dos registros, os principais atributos disponíveis e a forma como os valores foram carregados a partir da fonte.

Esta etapa funciona como uma primeira leitura do dataset antes da execução das validações sistemáticas de schema, completude, cardinalidade e integridade.

In [35]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Shipping Days
0,71,CA-2017-106075,18 de setembro de 2017,23 de setembro de 2017,Standard Class,HM-14980,Henry MacAllister,Consumer,United States,New York City,New York,10009.0,East,OFF-BI-10004654,Office Supplies,Binders,Avery Binding System Hidden Tab Executive Styl...,4.616,5
1,390,CA-2018-101798,11 de dezembro de 2018,15 de dezembro de 2018,Standard Class,MV-18190,Mike Vittorini,Consumer,United States,New York City,New York,10009.0,East,OFF-BI-10000050,Office Supplies,Binders,"Angle-D Binders with Locking Rings, Label Holders",23.360,4
2,538,CA-2016-147851,3 de dezembro de 2016,8 de dezembro de 2016,Standard Class,MP-17470,Mark Packer,Home Office,United States,New York City,New York,10009.0,East,OFF-BI-10004528,Office Supplies,Binders,Cardinal Poly Pocket Divider Pockets for Ring ...,10.752,5
3,573,CA-2018-164826,28 de dezembro de 2018,4 de janeiro de 2019,Standard Class,JF-15415,Jennifer Ferguson,Consumer,United States,New York City,New York,10024.0,East,OFF-BI-10001922,Office Supplies,Binders,Storex Dura Pro Binders,33.264,7
4,681,CA-2017-164511,19 de novembro de 2017,24 de novembro de 2017,Standard Class,DJ-13630,Doug Jacobs,Consumer,United States,New York City,New York,10024.0,East,OFF-BI-10003305,Office Supplies,Binders,Avery Hanging File Binders,14.352,5


### 2.2 — Schema e Tipos de Dados

O schema inicial é inspecionado para identificar os tipos atribuídos às variáveis durante a leitura do arquivo e a quantidade de valores não nulos em cada coluna.

Essa validação permite detectar antecipadamente variáveis cujo tipo técnico não representa corretamente sua semântica de negócio, além de fornecer uma primeira indicação de problemas de completude.

In [36]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9800 non-null   int64  
 1   Order ID       9800 non-null   str    
 2   Order Date     9800 non-null   str    
 3   Ship Date      9800 non-null   str    
 4   Ship Mode      9800 non-null   str    
 5   Customer ID    9800 non-null   str    
 6   Customer Name  9800 non-null   str    
 7   Segment        9800 non-null   str    
 8   Country        9800 non-null   str    
 9   City           9800 non-null   str    
 10  State          9800 non-null   str    
 11  Postal Code    9789 non-null   float64
 12  Region         9800 non-null   str    
 13  Product ID     9800 non-null   str    
 14  Category       9800 non-null   str    
 15  Sub-Category   9800 non-null   str    
 16  Product Name   9800 non-null   str    
 17  Sales          9800 non-null   float64
 18  Shipping Days  9800

### 2.3 — Inventário de Variáveis

O conjunto de colunas é registrado explicitamente para documentar os atributos disponíveis na fonte e apoiar as validações posteriores de entidades, chaves e granularidade.

O dataset contém informações relacionadas a pedidos, clientes, produtos, localização geográfica, vendas e logística de envio.

In [37]:
df.columns.tolist()

['Row ID',
 'Order ID',
 'Order Date',
 'Ship Date',
 'Ship Mode',
 'Customer ID',
 'Customer Name',
 'Segment',
 'Country',
 'City',
 'State',
 'Postal Code',
 'Region',
 'Product ID',
 'Category',
 'Sub-Category',
 'Product Name',
 'Sales',
 'Shipping Days']

### 2.4 — Cardinalidade e Completude

A cardinalidade de cada variável é analisada em conjunto com a quantidade de valores ausentes.

Essa combinação permite identificar potenciais identificadores, atributos categóricos, campos com baixa ou alta cardinalidade e variáveis que exigem investigação específica de completude.

A análise também fornece evidências iniciais para avaliar a granularidade do dataset e a consistência das futuras entidades analíticas.

In [38]:
cardinality = pd.DataFrame({
    "Column": df.columns,
    "Unique Values": [df[col].nunique() for col in df.columns],
    "Missing Values": [df[col].isna().sum() for col in df.columns]
})

cardinality

,Column,Unique Values,Missing Values
0,Row ID,9800,0
1,Order ID,4922,0
2,Order Date,1230,0
3,Ship Date,1326,0
4,Ship Mode,4,0
5,Customer ID,793,0
6,Customer Name,793,0
7,Segment,3,0
8,Country,1,0
9,City,529,0


### 2.5 — Diagnóstico Inicial de Granularidade

O dataset contém 9.800 registros e 4.922 pedidos únicos (Order ID), indicando que um mesmo pedido pode estar representado em múltiplas linhas.

O Row ID apresenta 9.800 valores únicos e será avaliado como possível identificador técnico de cada registro.

Essa diferença entre quantidade de registros e quantidade de pedidos indica preliminarmente que a granularidade da fonte está abaixo do nível do pedido. Essa característica deverá ser validada e considerada no cálculo das métricas de negócio, evitando interpretar cada linha como um pedido independente.

Também foram identificados pontos que requerem validação adicional:

- Customer ID: 793 valores únicos;
- Customer Name: 793 valores únicos;
- Product ID: 1.861 valores únicos;
- Product Name: 1.849 valores únicos;
- Postal Code: 11 valores ausentes;
- Order Date e Ship Date: armazenadas como texto.

Esses resultados orientam as verificações de integridade executadas nas próximas etapas.

## 3. Validação de Chaves e Integridade

Nesta etapa, são avaliadas possíveis chaves do dataset e a consistência das relações entre identificadores e seus respectivos atributos.

O objetivo é verificar se os identificadores observados representam adequadamente suas entidades e identificar restrições que deverão ser consideradas posteriormente na preparação e na modelagem analítica.

### 3.1 — Validação do Row ID

Row ID é avaliado como candidato a identificador técnico dos registros da fonte.

A validação compara a quantidade total de registros com a quantidade de valores únicos e verifica explicitamente a existência de identificadores duplicados.

In [39]:
total_rows = len(df)
unique_row_ids = df["Row ID"].nunique()
duplicated_row_ids = df["Row ID"].duplicated().sum()

print(f"Total de registros: {total_rows:,}")
print(f"Row IDs únicos: {unique_row_ids:,}")
print(f"Row IDs duplicados: {duplicated_row_ids:,}")

Total de registros: 9,800
Row IDs únicos: 9,800
Row IDs duplicados: 0


#### Resultado da Validação

O campo Row ID apresentou 9.800 valores únicos para 9.800 registros, sem ocorrência de duplicidades.

Dessa forma, Row ID pode ser considerado o identificador técnico único de cada registro no dataset original.

Essa conclusão não implica que cada registro represente um pedido distinto, pois existem apenas 4.922 Order ID únicos. A diferença reforça a necessidade de distinguir a unicidade técnica da granularidade de negócio.

### 3.2 — Validação da Entidade Cliente

A entidade Cliente é avaliada verificando se cada Customer ID mantém uma relação consistente com Customer Name e Segment.

Para que Customer ID possa representar uma chave adequada da futura dimensão de clientes, cada identificador deve estar associado a um único nome e a um único segmento no dataset analisado.

In [40]:
customer_name_check = (
    df.groupby("Customer ID")["Customer Name"]
      .nunique()
)

customer_name_check.value_counts().sort_index()

Customer Name
1    793
Name: count, dtype: int64

In [41]:
customer_segment_check = (
    df.groupby("Customer ID")["Segment"]
      .nunique()
)

customer_segment_check.value_counts().sort_index()

Segment
1    793
Name: count, dtype: int64

#### Resultado da Validação

Foram identificados 793 clientes distintos.

A análise de integridade demonstrou que cada Customer ID está associado a exatamente um Customer Name e a um único Segment.

Não foram identificadas inconsistências nessas relações.

Dessa forma, Customer ID pode ser utilizado como chave da futura dimensão de clientes (Dim_Cliente), composta inicialmente por:

- Customer ID
- Customer Name
- Segment

### 3.3 — Validação da Entidade Produto

A entidade Produto é avaliada para verificar se Product ID identifica de forma consistente um único produto e mantém relações estáveis com Product Name, Category e Sub-Category.

A diferença previamente observada entre 1.861 identificadores de produto e 1.849 nomes distintos exige uma investigação específica antes de assumir Product ID como chave única da futura dimensão de produtos.

A validação começa pela relação entre Product ID e Product Name e, posteriormente, verifica a consistência das classificações de categoria e subcategoria.

#### Investigação das Inconsistências de Produto

A distribuição anterior permite identificar quantos Product ID estão associados a mais de um Product Name.

Os identificadores que apresentam essa condição são isolados para verificar a extensão da inconsistência antes da análise detalhada dos respectivos registros.

In [42]:
product_name_check = (
    df.groupby("Product ID")["Product Name"]
      .nunique()
)

product_name_check.value_counts().sort_index()

Product Name
1    1829
2      32
Name: count, dtype: int64

In [43]:
inconsistent_product_ids = product_name_check[
    product_name_check > 1
].index

len(inconsistent_product_ids)

32

In [44]:
product_inconsistencies = (
    df[df["Product ID"].isin(inconsistent_product_ids)]
    [["Product ID", "Product Name", "Category", "Sub-Category"]]
    .drop_duplicates()
    .sort_values(["Product ID", "Product Name"])
)

product_inconsistencies

,Product ID,Product Name,Category,Sub-Category
6180,FUR-BO-10002213,DMI Eclipse Executive Suite Bookcases,Furniture,Bookcases
6146,FUR-BO-10002213,"Sauder Forest Hills Library, Woodland Oak Finish",Furniture,Bookcases
6843,FUR-CH-10001146,"Global Task Chair, Black",Furniture,Chairs
6477,FUR-CH-10001146,"Global Value Mid-Back Manager's Chair, Gray",Furniture,Chairs
8418,FUR-FU-10001473,DAX Wood Document Frame,Furniture,Furnishings
...,...,...,...,...
7859,TEC-PH-10002200,Samsung Galaxy Note 2,Technology,Phones
7648,TEC-PH-10002310,Panasonic KX T7731-B Digital phone,Technology,Phones
7908,TEC-PH-10002310,Plantronics Calisto P620-M USB Wireless Speake...,Technology,Phones
7912,TEC-PH-10004531,AT&T CL2909,Technology,Phones


In [45]:
product_classification_check = (
    df.groupby("Product ID")
      .agg(
          Product_Names=("Product Name", "nunique"),
          Categories=("Category", "nunique"),
          Subcategories=("Sub-Category", "nunique")
      )
)

product_classification_check[
    (product_classification_check["Categories"] > 1) |
    (product_classification_check["Subcategories"] > 1)
]

,Product_Names,Categories,Subcategories
Product ID,,,


#### Resultado da Validação

Foram identificados 1.861 Product ID distintos e 1.849 Product Name distintos.

A análise de integridade identificou que:

- 1.829 Product ID estão associados a exatamente um Product Name;
- 32 Product ID estão associados a dois Product Name distintos;
- os 32 casos representam produtos diferentes compartilhando o mesmo identificador;
- não foram identificadas inconsistências entre Product ID, Category e Sub-Category.

Portanto, embora Product ID seja consistente para a classificação em categoria e subcategoria, ele não identifica unicamente o produto no nível de Product Name.

Essa anomalia deverá ser considerada posteriormente na construção da Dim_Produto, evitando utilizar Product ID isoladamente como chave única sem uma estratégia de tratamento.

### 3.4 — Validação da Entidade Geografia

A entidade geográfica é avaliada a partir das relações entre Postal Code, City, State, Region e Country.

A auditoria possui dois objetivos principais: investigar os valores ausentes identificados em Postal Code e verificar se o código postal pode ser utilizado isoladamente como identificador geográfico.

Como Postal Code representa um identificador e não uma medida quantitativa, sua qualidade é analisada principalmente em termos de completude e consistência das relações com os demais atributos geográficos.

#### Investigação dos Códigos Postais Ausentes

Os registros sem Postal Code são isolados para identificar padrões geográficos comuns e verificar se existe, dentro do próprio dataset, alguma observação equivalente que possa fornecer evidência para uma eventual estratégia de tratamento.

A análise é realizada sem imputar ou alterar os valores originais.

In [46]:
missing_postal = df[df["Postal Code"].isna()][
    ["Row ID", "Order ID", "City", "State", "Postal Code", "Region", "Country"]
]

missing_postal

,Row ID,Order ID,City,State,Postal Code,Region,Country
2473,9387,US-2018-127292,Burlington,Vermont,NaN,East,United States
2474,9388,US-2018-127292,Burlington,Vermont,NaN,East,United States
3387,9149,US-2017-165505,Burlington,Vermont,NaN,East,United States
4021,9148,US-2017-165505,Burlington,Vermont,NaN,East,United States
4893,9389,US-2018-127292,Burlington,Vermont,NaN,East,United States
5503,9390,US-2018-127292,Burlington,Vermont,NaN,East,United States
6123,9742,CA-2016-117086,Burlington,Vermont,NaN,East,United States
6721,5275,CA-2016-162887,Burlington,Vermont,NaN,East,United States
7284,2235,CA-2018-104066,Burlington,Vermont,NaN,East,United States
7539,9147,US-2017-165505,Burlington,Vermont,NaN,East,United States


In [47]:
burlington_vermont = df[
    (df["City"] == "Burlington") &
    (df["State"] == "Vermont")
][
    ["City", "State", "Postal Code", "Region"]
]

burlington_vermont.value_counts(dropna=False)

City        State    Postal Code  Region
Burlington  Vermont  NaN          East      11
Name: count, dtype: int64

In [48]:
postal_geo_check = (
    df.dropna(subset=["Postal Code"])
      .groupby("Postal Code")
      .agg(
          Cities=("City", "nunique"),
          States=("State", "nunique"),
          Regions=("Region", "nunique"),
          Countries=("Country", "nunique")
      )
)

postal_geo_check[
    (postal_geo_check["Cities"] > 1) |
    (postal_geo_check["States"] > 1) |
    (postal_geo_check["Regions"] > 1) |
    (postal_geo_check["Countries"] > 1)
]

,Cities,States,Regions,Countries
Postal Code,,,,
92024.0,2,1,1,1


In [49]:
postal_92024 = df[
    df["Postal Code"] == 92024
][
    ["Row ID", "Order ID", "City", "State", "Postal Code", "Region", "Country"]
].sort_values("City")

postal_92024

,Row ID,Order ID,City,State,Postal Code,Region,Country
945,1114,CA-2018-140585,Encinitas,California,92024.0,West,United States
976,2820,CA-2017-160500,Encinitas,California,92024.0,West,United States
9038,1115,CA-2018-140585,Encinitas,California,92024.0,West,United States
9039,1116,CA-2018-140585,Encinitas,California,92024.0,West,United States
9076,2821,CA-2017-160500,Encinitas,California,92024.0,West,United States
802,1544,CA-2015-107181,San Diego,California,92024.0,West,United States
819,2948,CA-2018-169859,San Diego,California,92024.0,West,United States
876,6299,CA-2015-169852,San Diego,California,92024.0,West,United States
880,6701,CA-2018-145429,San Diego,California,92024.0,West,United States
1040,7919,CA-2016-152513,San Diego,California,92024.0,West,United States


#### Resultado da Validação

A análise identificou 626 códigos postais distintos, além de 11 registros com Postal Code ausente.

Os 11 valores ausentes estão concentrados exclusivamente em:

- City: Burlington
- State: Vermont
- Region: East
- Country: United States

Não existem outros registros de Burlington, Vermont, com código postal preenchido no dataset, não havendo evidência interna suficiente para realizar uma imputação baseada exclusivamente na própria fonte.

Também foi identificada uma exceção de integridade no Postal Code 92024, associado a duas cidades distintas (Encinitas e San Diego), embora State, Region e Country permaneçam consistentes.

Dessa forma, Postal Code isoladamente não deve ser assumido como chave geográfica única. A definição da chave da futura Dim_Geografia deverá considerar uma estratégia de modelagem que represente adequadamente a granularidade geográfica observada.

### 3.5 — Validação das Variáveis Temporais

As colunas Order Date e Ship Date foram carregadas como texto (str) e utilizam datas por extenso em português.

Antes de qualquer transformação definitiva, a auditoria verifica a padronização textual, a capacidade de conversão para datetime, a sequência cronológica entre pedido e envio e a consistência da variável derivada Shipping Days.

As conversões realizadas nesta seção possuem finalidade exclusivamente diagnóstica e não alteram o dataset bruto.

#### Inspeção Inicial das Variáveis Temporais

Uma amostra das colunas Order Date e Ship Date é inspecionada antes da aplicação das validações programáticas.

A inspeção permite observar a estrutura textual das datas e confirmar preliminarmente o uso do formato por extenso em português, que será posteriormente validado em todo o dataset.

In [50]:
df[["Order Date", "Ship Date"]].head(10)

,Order Date,Ship Date
0,18 de setembro de 2017,23 de setembro de 2017
1,11 de dezembro de 2018,15 de dezembro de 2018
2,3 de dezembro de 2016,8 de dezembro de 2016
3,28 de dezembro de 2018,4 de janeiro de 2019
4,19 de novembro de 2017,24 de novembro de 2017
5,23 de julho de 2018,28 de julho de 2018
6,10 de dezembro de 2015,15 de dezembro de 2015
7,30 de dezembro de 2018,3 de janeiro de 2019
8,15 de setembro de 2017,20 de setembro de 2017
9,4 de abril de 2016,8 de abril de 2016


#### Validação do Padrão Textual

Após a inspeção inicial, uma expressão regular é aplicada a todos os registros de Order Date e Ship Date para verificar se os valores seguem a estrutura esperada DD de mês de YYYY.

Essa validação transforma a observação visual da amostra em um controle sistemático sobre todo o dataset, permitindo identificar eventuais registros fora do padrão antes da conversão para datetime.

In [51]:
order_date_pattern = df["Order Date"].str.contains(
    r"^\d{1,2} de [A-Za-zÀ-ÿ]+ de \d{4}$",
    regex=True,
    na=False
)

ship_date_pattern = df["Ship Date"].str.contains(
    r"^\d{1,2} de [A-Za-zÀ-ÿ]+ de \d{4}$",
    regex=True,
    na=False
)

print(f"Order Date no padrão esperado: {order_date_pattern.sum():,} de {len(df):,}")
print(f"Ship Date no padrão esperado: {ship_date_pattern.sum():,} de {len(df):,}")

Order Date no padrão esperado: 9,800 de 9,800
Ship Date no padrão esperado: 9,800 de 9,800


#### Estratégia de Conversão

Como as datas utilizam nomes de meses em português, é definido um mapeamento entre cada mês e sua representação numérica.

Essa normalização permite transformar a estrutura textual original em um formato compatível com a conversão para `datetime`, sem alterar as colunas originais durante esta etapa de auditoria.

In [52]:
month_map = {
    "janeiro": "01",
    "fevereiro": "02",
    "março": "03",
    "abril": "04",
    "maio": "05",
    "junho": "06",
    "julho": "07",
    "agosto": "08",
    "setembro": "09",
    "outubro": "10",
    "novembro": "11",
    "dezembro": "12"
}

#### Conversão Controlada para Datetime

A função abaixo encapsula a regra de conversão das datas, aplicando o mapeamento dos meses, removendo os conectores textuais " de " e convertendo o resultado para datetime.

A transformação é realizada sobre uma cópia da série recebida. O parâmetro errors="coerce" funciona como mecanismo de controle: valores incompatíveis com o formato esperado são convertidos para NaT, permitindo que eventuais falhas sejam identificadas e quantificadas posteriormente.

In [53]:
def convert_portuguese_date(series):
    converted = series.copy()

    for month_name, month_number in month_map.items():
        converted = converted.str.replace(
            month_name,
            month_number,
            regex=False
        )

    converted = converted.str.replace(" de ", " ", regex=False)

    return pd.to_datetime(
        converted,
        format="%d %m %Y",
        errors="coerce"
    )

#### Validação da Conversão

As colunas Order Date e Ship Date são convertidas em estruturas auxiliares utilizando a regra definida anteriormente.

A quantidade de valores NaT resultantes é utilizada como controle de qualidade da conversão. Dessa forma, é possível verificar se algum registro originalmente presente na fonte não pôde ser interpretado pelo formato temporal definido.

In [54]:
order_date_test = convert_portuguese_date(df["Order Date"])
ship_date_test = convert_portuguese_date(df["Ship Date"])

print(f"Order Date inválidas: {order_date_test.isna().sum():,}")
print(f"Ship Date inválidas: {ship_date_test.isna().sum():,}")

Order Date inválidas: 0
Ship Date inválidas: 0


#### Validação da Sequência Temporal

Após confirmar a conversão das datas, é verificada a relação cronológica entre pedido e envio.

Para cada registro, Ship Date deve ser igual ou posterior a Order Date. A existência de uma data de envio anterior à data do pedido indicaria uma inconsistência temporal que exigiria investigação.

In [55]:
invalid_shipping_sequence = ship_date_test < order_date_test

print(
    f"Envios anteriores ao pedido: "
    f"{invalid_shipping_sequence.sum():,}"
)

Envios anteriores ao pedido: 0


#### Validação de Shipping Days

A consistência da variável Shipping Days é verificada recalculando o intervalo, em dias, entre Ship Date e Order Date.

O valor calculado é comparado com o valor armazenado na fonte para identificar eventuais divergências entre a informação temporal original e a duração de envio registrada.

In [56]:
calculated_shipping_days = (
    ship_date_test - order_date_test
).dt.days

shipping_days_check = calculated_shipping_days != df["Shipping Days"]

print(
    f"Inconsistências em Shipping Days: "
    f"{shipping_days_check.sum():,}"
)

Inconsistências em Shipping Days: 0


#### Resultado da Validação

As colunas Order Date e Ship Date estão armazenadas originalmente como texto (str) no formato por extenso em português.

A auditoria identificou que:

- 100% dos 9.800 registros de Order Date seguem o padrão textual esperado;
- 100% dos 9.800 registros de Ship Date seguem o padrão textual esperado;
- nenhuma data inválida foi identificada durante a conversão de teste para datetime;
- não existem registros em que Ship Date seja anterior a Order Date;
- a coluna Shipping Days corresponde integralmente à diferença calculada entre Ship Date e Order Date.

Portanto, não foram identificadas inconsistências temporais. As colunas de data necessitam apenas da conversão definitiva para datetime na etapa de preparação dos dados, antes das análises temporais e da futura construção da Dim_Calendario.

## 4. Validação de Duplicidades e Granularidade

Nesta etapa, são investigadas possíveis duplicidades no dataset e a distribuição das linhas entre os pedidos.

A análise distingue a unicidade técnica fornecida por Row ID da duplicidade baseada nos atributos de negócio e, posteriormente, quantifica a granularidade efetiva dos registros.

### 4.1 — Duplicidade Integral

A primeira verificação considera simultaneamente todas as colunas do dataset para identificar registros integralmente idênticos.

Como Row ID também participa dessa comparação, este controle verifica a existência de duplicidades completas no nível técnico da fonte antes da aplicação de uma regra específica de duplicidade de negócio.

In [57]:
full_duplicates = df.duplicated().sum()

print(f"Linhas completamente duplicadas: {full_duplicates:,}")

Linhas completamente duplicadas: 0


### 4.2 — Duplicidade de Negócio

A ausência de registros integralmente duplicados não elimina a possibilidade de duplicidades no nível de negócio, pois Row ID representa um identificador técnico único da fonte.

Por esse motivo, uma segunda validação exclui Row ID da comparação e considera os demais atributos como representação do evento de negócio.

O parâmetro keep=False é utilizado para identificar todas as ocorrências envolvidas em possíveis duplicidades, permitindo que os registros sejam posteriormente isolados e investigados.

In [58]:
business_columns = [
    col for col in df.columns
    if col != "Row ID"
]

business_duplicates = df.duplicated(
    subset=business_columns,
    keep=False
)

print(
    f"Registros envolvidos em possíveis duplicidades de negócio: "
    f"{business_duplicates.sum():,}"
)

Registros envolvidos em possíveis duplicidades de negócio: 2


### 4.3 — Investigação do Caso Identificado

A validação anterior identificou dois registros envolvidos em uma possível duplicidade de negócio.

Essas ocorrências são isoladas e ordenadas por Order ID, Product ID e Sales para permitir a comparação direta de seus atributos e verificar se existe alguma diferença de negócio que justifique a presença de ambas as linhas.

A classificação definitiva como duplicidade de negócio será realizada somente após essa inspeção.

In [59]:
possible_business_duplicates = (
    df[business_duplicates]
    .sort_values(
        ["Order ID", "Product ID", "Sales"]
    )
)

possible_business_duplicates

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Shipping Days
6623,3406,US-2015-150119,23 de abril de 2015,27 de abril de 2015,Standard Class,LB-16795,Laurel Beltran,Home Office,United States,Columbus,Ohio,43229.0,East,FUR-CH-10002965,Furniture,Chairs,Global Leather Highback Executive Chair with P...,281.372,4
6624,3407,US-2015-150119,23 de abril de 2015,27 de abril de 2015,Standard Class,LB-16795,Laurel Beltran,Home Office,United States,Columbus,Ohio,43229.0,East,FUR-CH-10002965,Furniture,Chairs,Global Leather Highback Executive Chair with P...,281.372,4


#### Contextualização do Pedido

Para complementar a investigação, o pedido US-2015-150119 é analisado integralmente.

A inspeção permite verificar os registros associados ao mesmo Order ID e confirmar se as duas ocorrências suspeitas representam linhas distintas do pedido ou uma repetição do mesmo evento de negócio.

In [60]:
order_150119 = df[
    df["Order ID"] == "US-2015-150119"
][
    ["Row ID", "Order ID", "Product ID", "Product Name", "Sales"]
].sort_values("Row ID")

order_150119

,Row ID,Order ID,Product ID,Product Name,Sales
6623,3406,US-2015-150119,FUR-CH-10002965,Global Leather Highback Executive Chair with P...,281.372
6624,3407,US-2015-150119,FUR-CH-10002965,Global Leather Highback Executive Chair with P...,281.372
152,3408,US-2015-150119,OFF-BI-10000145,Zipper Ring Binder Pockets,7.488
8741,3409,US-2015-150119,FUR-FU-10002191,G.E. Halogen Desk Lamp Bulbs,22.336


#### Resultado da Validação

Não foram identificadas linhas completamente duplicadas considerando todas as 19 colunas do dataset.

Ao excluir Row ID da comparação, foram identificados dois registros com atributos de negócio integralmente idênticos, correspondentes aos Row ID 3406 e 3407.

A investigação do pedido US-2015-150119 confirmou que ambas as ocorrências estão associadas ao mesmo:

- Order ID;
- Product ID;
- Product Name;
- valor de Sales;
- cliente;
- período de pedido e envio;
- localização;
- demais atributos de negócio avaliados.

A única diferença entre os dois registros é o identificador técnico Row ID.

Com base na regra definida nesta auditoria, os registros são classificados como uma duplicidade de negócio. Na etapa de preparação dos dados, uma das ocorrências deverá ser removida, preservando apenas um registro representativo do evento e mantendo a fonte original em data/raw inalterada.

### 4.4 — Distribuição de Linhas por Pedido

Após a investigação das duplicidades, a quantidade de registros associados a cada Order ID é calculada para caracterizar formalmente a granularidade da fonte.

A distribuição permite avaliar quantas linhas compõem cada pedido e observar medidas como média, mediana, quartis e quantidade máxima de linhas por Order ID.

Essa análise é necessária para confirmar se os registros representam pedidos consolidados ou itens individuais associados a um mesmo pedido.

In [61]:
lines_per_order = (
    df.groupby("Order ID")
      .size()
)

lines_per_order.describe()

count    4922.000000
mean        1.991061
std         1.406115
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max        14.000000
dtype: float64

#### Segmentação por Estrutura do Pedido

Para complementar a distribuição estatística, os pedidos são classificados de acordo com a quantidade de linhas associadas a cada Order ID.

A segmentação distingue pedidos representados por uma única linha daqueles compostos por múltiplas linhas, permitindo quantificar a participação de cada estrutura no dataset.

In [62]:
single_line_orders = (lines_per_order == 1).sum()
multiple_line_orders = (lines_per_order > 1).sum()

print(f"Pedidos com 1 linha: {single_line_orders:,}")
print(f"Pedidos com múltiplas linhas: {multiple_line_orders:,}")

print(
    f"Percentual com 1 linha: "
    f"{single_line_orders / len(lines_per_order):.2%}"
)

print(
    f"Percentual com múltiplas linhas: "
    f"{multiple_line_orders / len(lines_per_order):.2%}"
)

Pedidos com 1 linha: 2,499
Pedidos com múltiplas linhas: 2,423
Percentual com 1 linha: 50.77%
Percentual com múltiplas linhas: 49.23%


### 4.5 — Diagnóstico da Granularidade dos Pedidos

O dataset contém 9.800 linhas correspondentes a 4.922 pedidos distintos.

A análise da distribuição das linhas por pedido demonstrou que:

- 2.499 pedidos (50,77%) possuem apenas uma linha;
- 2.423 pedidos (49,23%) possuem múltiplas linhas;
- a média é de aproximadamente 1,99 linhas por pedido;
- a mediana é de 1 linha;
- 75% dos pedidos possuem até 2 linhas;
- o maior pedido possui 14 linhas.

Esses resultados confirmam que a granularidade do dataset está no nível da linha de venda associada ao pedido, e não no nível do pedido consolidado.

Consequentemente, métricas relacionadas a pedidos devem respeitar essa granularidade. A quantidade de pedidos deverá ser calculada utilizando Order ID distinto, evitando interpretar a quantidade total de linhas como número de pedidos.

Da mesma forma, o ticket médio deverá ser calculado no nível do pedido, agregando previamente as vendas das linhas pertencentes ao mesmo Order ID. A média direta da coluna Sales representaria o valor médio por linha de venda, e não o ticket médio por pedido.

## 5. Consolidação da Auditoria

### 5.1 — Principais Resultados

A auditoria foi realizada sobre o dataset original contendo 9.800 registros e 19 colunas.

**Completude**

- 18 das 19 colunas não apresentam valores ausentes.
- Postal Code possui 11 valores ausentes, todos associados a Burlington, Vermont.

**Granularidade**

- Foram identificados 4.922 pedidos distintos em 9.800 linhas.
- 50,77% dos pedidos possuem uma única linha e 49,23% possuem múltiplas linhas.
- A fonte encontra-se no nível da linha de venda associada ao pedido, e não no nível do pedido consolidado.

**Identificação dos registros**

- Row ID é único nos 9.800 registros.
- Foi identificada uma duplicidade de negócio composta por dois registros que diferem exclusivamente pelo Row ID.

**Clientes**

- Foram identificados 793 clientes.
- Cada Customer ID está associado consistentemente a um único Customer Name e Segment.

**Produtos**

- Foram identificados 1.861 Product ID.
- 32 identificadores estão associados a dois Product Name distintos.
- Category e Sub-Category permanecem consistentes para cada Product ID.
- Product ID isoladamente não garante unicidade no nível de Product Name.

**Geografia**

- Foram identificados 626 códigos postais preenchidos distintos.
- Existem 11 valores ausentes de Postal Code, concentrados em Burlington, Vermont.
- O código postal 92024 aparece associado a duas cidades: Encinitas e San Diego.
- Postal Code isoladamente não deve ser considerado uma chave geográfica única.

**Variáveis temporais**

- Order Date e Ship Date estão armazenadas originalmente como texto em português.
- Todos os 9.800 registros de cada coluna seguem o padrão esperado e são convertíveis para datetime.
- Não existem envios anteriores à data do pedido.
- Shipping Days é integralmente consistente com a diferença entre Ship Date e Order Date.

### 5.2 — Regras Definidas para Tratamento

Com base nas evidências obtidas durante a auditoria, a etapa de preparação deverá seguir as seguintes regras:

1. preservar o dataset original em data/raw;
2. converter Order Date e Ship Date para datetime;
3. tratar Postal Code como identificador geográfico, e não como medida numérica;
4. preservar os 11 valores ausentes de Postal Code, uma vez que não existe evidência interna suficiente para uma imputação confiável;
5. preservar a inconsistência observada entre Product ID e Product Name até a definição de uma estratégia de modelagem;
6. definir uma chave adequada para a futura Dim_Produto;
7. definir uma estratégia de chave para a futura Dim_Geografia;
8. remover a duplicidade de negócio identificada, preservando apenas uma ocorrência;
9. preservar a granularidade de linha de venda na futura tabela fato;
10. calcular métricas relacionadas a pedidos utilizando Order ID distinto e respeitando a granularidade identificada.

As questões relacionadas a produtos e geografia são mantidas como decisões de modelagem posteriores, evitando alterações arbitrárias na fonte durante a limpeza básica.

### 5.3 — Encaminhamento para a Preparação dos Dados

A auditoria estabeleceu as regras necessárias para transformar a fonte bruta de forma controlada e reproduzível.

Os tratamentos determinísticos identificados nesta etapa serão centralizados no módulo src/data_cleaning.py e executados no notebook 02_data_preparation.ipynb, mantendo separadas as responsabilidades de diagnóstico, transformação e validação.

O dataset original permanecerá inalterado em data/raw, enquanto o resultado aprovado pelos controles de qualidade será persistido em data/processed para utilização nas etapas analíticas subsequentes.

As decisões relacionadas à estrutura das futuras dimensões, especialmente Dim_Produto, Dim_Geografia e Dim_Calendario, serão tratadas nas etapas de modelagem apropriadas, com base nas restrições identificadas nesta auditoria.